# Visual Genome Caption Generation
## Task 1 Training: Object & Attribute Classification

Train object classifier and attribute classifier jointly.

In [ ]:
# Imports
import os
import sys
from pathlib import Path

# Add src to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src.utils import load_config, Logger, CheckpointManager
from src.data import ObjectAttributeDataset, build_task1_datasets
from src.models.task1 import ObjectClassifier, AttributeClassifier
from src.training import Task1Trainer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name()}")

In [ ]:
# Load configuration
config = load_config("../configs/task1_config.yaml")
print("Task 1 Configuration:")
print(f"- Backbone: {config.backbone.name}")
print(f"- Object classes: {config.labels.max_objects}")
print(f"- Attribute classes: {config.labels.max_attributes}")
print(f"- Batch size: {config.training.batch_size}")
print(f"- Max epochs: {config.training.max_epochs}")
print(f"- Learning rate: {config.training.lr}")

In [ ]:
# Create datasets
print("Creating Task 1 datasets...")

# Check if processed data exists
processed_dir = Path("../data/processed/task1")
if not processed_dir.exists():
    raise FileNotFoundError("Processed data not found. Run notebook 02_data_processing.ipynb first")

train_ds, val_ds, test_ds = build_task1_datasets(
    processed_dir=str(processed_dir),
    image_dir="../data/raw/images",
    roi_size=config.image.roi_size,
    use_feature_cache=config.dataset.use_feature_cache,
    max_samples=None  # Use all samples
)

print(f"Train: {len(train_ds)} samples")
print(f"Val: {len(val_ds)} samples")
print(f"Test: {len(test_ds)} samples")

# Create data loaders
train_loader = DataLoader(
    train_ds,
    batch_size=config.training.batch_size,
    shuffle=True,
    num_workers=config.training.num_workers,
    pin_memory=config.training.pin_memory
)

val_loader = DataLoader(
    val_ds,
    batch_size=config.training.batch_size,
    shuffle=False,
    num_workers=config.training.num_workers,
    pin_memory=config.training.pin_memory
)

print("Data loaders created successfully")

In [ ]:
# Create models
print("Creating Task 1 models...")

device = config.device if torch.cuda.is_available() else "cpu"

# Object classifier
object_model = ObjectClassifier(
    num_classes=config.labels.max_objects,
    feature_dim=config.backbone.feature_dim,
    hidden_dim=config.model.object_hidden_dim,
    dropout=config.model.object_dropout,
    device=device
)

# Attribute classifier
attribute_model = AttributeClassifier(
    num_attributes=config.labels.max_attributes,
    feature_dim=config.backbone.feature_dim,
    hidden_dim=config.model.attribute_hidden_dim,
    dropout=config.model.attribute_dropout,
    device=device
)

print(f"Object model: {object_model.summary()}")
print(f"Attribute model: {attribute_model.summary()}")

# Create optimizers
object_optimizer = torch.optim.AdamW(
    object_model.parameters(),
    lr=config.training.lr,
    weight_decay=config.optimizer.weight_decay
)

attribute_optimizer = torch.optim.AdamW(
    attribute_model.parameters(),
    lr=config.training.lr,
    weight_decay=config.optimizer.weight_decay
)

print("Models and optimizers created successfully")

In [ ]:
# Setup logging and checkpointing
experiment_name = f"task1_{config.backbone.name}_{config.training.lr:.0e}"

logger = Logger(
    log_dir="../logs",
    experiment_name=experiment_name,
    use_tensorboard=True,
    use_wandb=False  # Set to True if you have wandb
)

checkpoint_manager = CheckpointManager(
    checkpoint_dir="../checkpoints",
    experiment_name=experiment_name,
    max_checkpoints=3,
    save_best_only=False
)

print(f"Experiment: {experiment_name}")
print(f"Logging to: ../logs/{experiment_name}")
print(f"Checkpoints to: ../checkpoints/{experiment_name}")

In [ ]:
# Create trainer
trainer = Task1Trainer(
    object_model=object_model,
    attribute_model=attribute_model,
    train_loader=train_loader,
    val_loader=val_loader,
    object_optimizer=object_optimizer,
    attribute_optimizer=attribute_optimizer,
    device=device,
    logger=logger,
    checkpoint_manager=checkpoint_manager,
    # Training config
    max_epochs=config.training.max_epochs,
    early_stopping_patience=config.training.early_stopping_patience,
    gradient_clip_val=config.training.gradient_clip_val,
    log_every_n_steps=config.training.log_every_n_steps,
    # Loss weights
    object_weight=config.loss.object_weight,
    attribute_weight=config.loss.attribute_weight,
    attribute_pos_weight=torch.tensor([config.loss.attribute_pos_weight] * config.labels.max_attributes)
)

print("Trainer created successfully")
print(f"Training for {config.training.max_epochs} epochs on {device}")

# Start training
print("\n" + "="*50)
print("STARTING TRAINING")
print("="*50)

final_metrics = trainer.train()

print("\n" + "="*50)
print("TRAINING COMPLETED")
print("="*50)
print("Final validation metrics:")
for key, value in final_metrics.items():
    print(".4f")

In [ ]:
# Test trained models on a few examples
print("\nTesting trained models...")

# Load best checkpoint
best_checkpoint = checkpoint_manager.get_best_checkpoint_path()
if best_checkpoint:
    print(f"Loading best checkpoint: {best_checkpoint}")
    trainer.resume_from_checkpoint(best_checkpoint)

# Test on validation set
test_loader = DataLoader(val_ds, batch_size=4, shuffle=False)
trainer.model.eval()

with torch.no_grad():
    for i, batch in enumerate(test_loader):
        if i >= 3:  # Test only first 3 batches
            break

        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

        # Get predictions
        features = batch['feature'] if 'feature' in batch else batch['image']
        object_preds = object_model.predict(features)
        attribute_preds = attribute_model.predict(features)

        print(f"\nBatch {i+1}:")
        print(f"Object predictions: {object_preds[:4].tolist()}")
        print(f"Attribute predictions shape: {attribute_preds.shape}")

print("\nTraining notebook completed!")
print("Check ../logs/ and ../checkpoints/ for results")